# audio_01_build_manifest_es — Construcción del manifest (ES)

Este notebook genera el archivo `manifest_es.csv` para entrenamiento de emociones en audio (español).  
Recorre y normaliza varios subconjuntos/datasets (MESD ES, MESD ES Embedded y opcionalmente SES-SD) y exporta un CSV con rutas y etiquetas listas para usar en los notebooks de entrenamiento.

**Salida:** `data/audio/manifest_es.csv`

## 1) Rutas del proyecto y verificación de carpetas

Definimos la raíz del proyecto y las rutas a las carpetas de audio en `data/audio/raw`.  
Comprobamos si existen los directorios esperados para evitar errores antes de construir el manifest.

In [1]:
import re
from pathlib import Path
import pandas as pd

PROJECT_ROOT = Path("..").resolve()
RAW = PROJECT_ROOT / "data" / "audio" / "raw"
OUT_DIR = PROJECT_ROOT / "data" / "audio"
OUT_DIR.mkdir(parents=True, exist_ok=True)

MESD_ES_DIR = RAW / "mesd_es"               # wav sueltos
MESD_EMB_DIR = RAW / "mesd_es_emb"          # Level1 / Level2
SES_SD_DIR = RAW / "ses_sd"                 # si lo mueves aquí (opcional)

print("PROJECT_ROOT:", PROJECT_ROOT)
print("MESD_ES_DIR exists:", MESD_ES_DIR.exists(), MESD_ES_DIR)
print("MESD_EMB_DIR exists:", MESD_EMB_DIR.exists(), MESD_EMB_DIR)
print("SES_SD_DIR exists:", SES_SD_DIR.exists(), SES_SD_DIR)

PROJECT_ROOT: C:\Users\Rafa\Downloads\proyecto-ia-20260212T103152Z-3-001\proyecto-ia\multimodal-emocion
MESD_ES_DIR exists: True C:\Users\Rafa\Downloads\proyecto-ia-20260212T103152Z-3-001\proyecto-ia\multimodal-emocion\data\audio\raw\mesd_es
MESD_EMB_DIR exists: True C:\Users\Rafa\Downloads\proyecto-ia-20260212T103152Z-3-001\proyecto-ia\multimodal-emocion\data\audio\raw\mesd_es_emb
SES_SD_DIR exists: True C:\Users\Rafa\Downloads\proyecto-ia-20260212T103152Z-3-001\proyecto-ia\multimodal-emocion\data\audio\raw\ses_sd


## 2) MESD ES — Parseo de nombres y creación del manifest

Creamos un mapeo de emociones a etiquetas estándar (por ejemplo, `happy → joy`, `sad → sadness`).  
Luego parseamos el nombre del archivo para extraer:
- `label` (emoción)
- `speaker` aproximado (según partes del nombre)
- `utt_id` (identificador de la utterance)

Con ello construimos `mesd_df` (un DataFrame con una fila por audio).

In [2]:
# Mapeo emociones MESD -> estándar
MESD_MAP = {
    "anger": "anger",
    "disgust": "disgust",
    "fear": "fear",
    "happy": "joy",
    "happiness": "joy",
    "joy": "joy",
    "neutral": "neutral",
    "sad": "sadness",
    "sadness": "sadness",
    "surprise": "surprise",
}

def parse_mesd_filename(name: str):
    """
    Ej: Anger_C_A_arriba.wav
    Devuelve (label, speaker, utt) o None si no encaja
    """
    stem = Path(name).stem
    parts = stem.split("_")
    if len(parts) < 2:
        return None
    emo = parts[0].strip().lower()
    label = MESD_MAP.get(emo, None)
    if label is None:
        return None

    # speaker aproximado: las 2-3 siguientes partes si existen (C_A, etc.)
    speaker = "_".join(parts[1:3]) if len(parts) >= 3 else parts[1]
    utt = stem
    return label, speaker, utt

mesd_rows = []
for wav in sorted(MESD_ES_DIR.glob("*.wav")):
    parsed = parse_mesd_filename(wav.name)
    if not parsed:
        continue
    label, speaker, utt = parsed
    mesd_rows.append({
        "path": str(wav),
        "label": label,
        "dataset": "mesd_es",
        "speaker": f"mesd_{speaker}",
        "split": "all",
        "utt_id": utt
    })

mesd_df = pd.DataFrame(mesd_rows)
print("MESD rows:", len(mesd_df))
print(mesd_df["label"].value_counts())
mesd_df.head()

MESD rows: 862
label
disgust    144
fear       144
joy        144
sadness    144
anger      143
neutral    143
Name: count, dtype: int64


,path,label,dataset,speaker,split,utt_id
0,C:\Users\Rafa\Downloads\proyecto-ia-20260212T1...,anger,mesd_es,mesd_C_A,all,Anger_C_A_abajo
1,C:\Users\Rafa\Downloads\proyecto-ia-20260212T1...,anger,mesd_es,mesd_C_A,all,Anger_C_A_adios
2,C:\Users\Rafa\Downloads\proyecto-ia-20260212T1...,anger,mesd_es,mesd_C_A,all,Anger_C_A_antes
3,C:\Users\Rafa\Downloads\proyecto-ia-20260212T1...,anger,mesd_es,mesd_C_A,all,Anger_C_A_arriba
4,C:\Users\Rafa\Downloads\proyecto-ia-20260212T1...,anger,mesd_es,mesd_C_A,all,Anger_C_A_ayer


## 3) MESD ES Embedded — Level1/Level2 (speaker y nivel)

Procesamos la variante "embedded" (carpetas `Level1` / `Level2`).  
Aquí extraemos además el nivel (`L1` o `L2`) y creamos un `speaker` más informativo (género, grupo y nivel).

Resultado: `emb_df` con columnas extra como `level`.

In [3]:
def parse_mesd_emb_filename(name: str):
    """
    Ej: Anger_F_B_L1_abuso.wav
    """
    stem = Path(name).stem
    parts = stem.split("_")
    if len(parts) < 4:
        return None
    
    emo = parts[0].strip().lower()
    label = MESD_MAP.get(emo, None)
    if label is None:
        return None

    gender = parts[1].upper()  # F/M?
    spk_grp = parts[2].upper() # A/B/C?
    lvl = parts[3].upper()     # L1/L2
    speaker = f"{gender}_{spk_grp}_{lvl}"
    utt = stem
    return label, speaker, utt, lvl

emb_rows = []
for level_dir in [MESD_EMB_DIR / "Level1", MESD_EMB_DIR / "Level2"]:
    if not level_dir.exists():
        continue
    for wav in sorted(level_dir.glob("*.wav")):
        parsed = parse_mesd_emb_filename(wav.name)
        if not parsed:
            continue
        label, speaker, utt, lvl = parsed
        emb_rows.append({
            "path": str(wav),
            "label": label,
            "dataset": "mesd_es_emb",
            "speaker": f"mesdemb_{speaker}",
            "split": "all",
            "utt_id": utt,
            "level": lvl
        })

emb_df = pd.DataFrame(emb_rows)
print("MESD-EMB rows:", len(emb_df))
if len(emb_df):
    print(emb_df["label"].value_counts())
emb_df.head() if len(emb_df) else emb_df

MESD-EMB rows: 288
label
anger      48
disgust    48
fear       48
joy        48
neutral    48
sadness    48
Name: count, dtype: int64


,path,label,dataset,speaker,split,utt_id,level
0,C:\Users\Rafa\Downloads\proyecto-ia-20260212T1...,anger,mesd_es_emb,mesdemb_F_B_L1,all,Anger_F_B_L1_abuso,L1
1,C:\Users\Rafa\Downloads\proyecto-ia-20260212T1...,anger,mesd_es_emb,mesdemb_F_B_L1,all,Anger_F_B_L1_amenazado,L1
2,C:\Users\Rafa\Downloads\proyecto-ia-20260212T1...,anger,mesd_es_emb,mesdemb_F_B_L1,all,Anger_F_B_L1_amenazador,L1
3,C:\Users\Rafa\Downloads\proyecto-ia-20260212T1...,anger,mesd_es_emb,mesdemb_F_B_L1,all,Anger_F_B_L1_ataque,L1
4,C:\Users\Rafa\Downloads\proyecto-ia-20260212T1...,anger,mesd_es_emb,mesdemb_F_B_L1,all,Anger_F_B_L1_atasco,L1


## 4) (Opcional) SES-SD — Añadir otro dataset si está disponible

Si existe la carpeta `ses_sd`, recorremos sus WAVs y extraemos información desde el nombre:
- speaker y grupo
- emoción (código tipo `SAD`, `ANG`, etc.)
- idioma (`SPA`)

Se construye `sessd_df`. Si no existe la carpeta, este bloque no añade filas.

In [4]:
SES_MAP = {
    "ANG": "anger",
    "DIS": "disgust",
    "FEA": "fear",
    "HAP": "joy",
    "NEU": "neutral",
    "SAD": "sadness",
    "SUR": "surprise",
}

ses_rows = []
if SES_SD_DIR.exists():
    for wav in sorted(SES_SD_DIR.glob("*.wav")):
        stem = wav.stem
        # ejemplo: 10_DFA_SAD_SPA
        parts = stem.split("_")
        if len(parts) < 4:
            continue
        spk = parts[0]          # 10
        group = parts[1]        # DFA/IEO/...
        emo = parts[2].upper()  # SAD/ANG/...
        lang = parts[3].upper() # SPA
        label = SES_MAP.get(emo)
        if not label:
            continue
        ses_rows.append({
            "path": str(wav),
            "label": label,
            "dataset": "ses_sd",
            "speaker": f"sessd_{spk}_{group}",
            "split": "all",
            "utt_id": stem,
            "lang": lang
        })

sessd_df = pd.DataFrame(ses_rows)
print("SES-SD rows:", len(sessd_df))
if len(sessd_df):
    print(sessd_df["label"].value_counts())
sessd_df.head() if len(sessd_df) else sessd_df

SES-SD rows: 1588
label
anger      265
disgust    265
fear       265
sadness    265
joy        264
neutral    264
Name: count, dtype: int64


,path,label,dataset,speaker,split,utt_id,lang
0,C:\Users\Rafa\Downloads\proyecto-ia-20260212T1...,anger,ses_sd,sessd_10_DFA,all,10_DFA_ANG_SPA,SPA
1,C:\Users\Rafa\Downloads\proyecto-ia-20260212T1...,disgust,ses_sd,sessd_10_DFA,all,10_DFA_DIS_SPA,SPA
2,C:\Users\Rafa\Downloads\proyecto-ia-20260212T1...,fear,ses_sd,sessd_10_DFA,all,10_DFA_FEA_SPA,SPA
3,C:\Users\Rafa\Downloads\proyecto-ia-20260212T1...,joy,ses_sd,sessd_10_DFA,all,10_DFA_HAP_SPA,SPA
4,C:\Users\Rafa\Downloads\proyecto-ia-20260212T1...,neutral,ses_sd,sessd_10_DFA,all,10_DFA_NEU_SPA,SPA


## 5) Unificar datasets y exportar `manifest_es.csv`

Unimos los DataFrames disponibles (`mesd_df`, `emb_df` y opcionalmente `sessd_df`) en un único `manifest_es`.  
Después exportamos el CSV final que usarán los notebooks de entrenamiento.

In [5]:
dfs = [mesd_df]

# añade emb si existe
if len(emb_df):
    dfs.append(emb_df)

# añade ses_sd si existe y quieres
if 'sessd_df' in globals() and len(sessd_df):
    dfs.append(sessd_df)  # descomenta si lo quieres mezclar también
    pass

manifest_es = pd.concat(dfs, ignore_index=True)

MANIFEST_ES = OUT_DIR / "manifest_es.csv"
manifest_es.to_csv(MANIFEST_ES, index=False)

print("✅ Guardado:", MANIFEST_ES)
print("Total:", len(manifest_es))
print(manifest_es["dataset"].value_counts())
print(manifest_es["label"].value_counts())
manifest_es.head()

✅ Guardado: C:\Users\Rafa\Downloads\proyecto-ia-20260212T103152Z-3-001\proyecto-ia\multimodal-emocion\data\audio\manifest_es.csv
Total: 2738
dataset
ses_sd         1588
mesd_es         862
mesd_es_emb     288
Name: count, dtype: int64
label
disgust    457
fear       457
sadness    457
anger      456
joy        456
neutral    455
Name: count, dtype: int64


,path,label,dataset,speaker,split,utt_id,level,lang
0,C:\Users\Rafa\Downloads\proyecto-ia-20260212T1...,anger,mesd_es,mesd_C_A,all,Anger_C_A_abajo,NaN,NaN
1,C:\Users\Rafa\Downloads\proyecto-ia-20260212T1...,anger,mesd_es,mesd_C_A,all,Anger_C_A_adios,NaN,NaN
2,C:\Users\Rafa\Downloads\proyecto-ia-20260212T1...,anger,mesd_es,mesd_C_A,all,Anger_C_A_antes,NaN,NaN
3,C:\Users\Rafa\Downloads\proyecto-ia-20260212T1...,anger,mesd_es,mesd_C_A,all,Anger_C_A_arriba,NaN,NaN
4,C:\Users\Rafa\Downloads\proyecto-ia-20260212T1...,anger,mesd_es,mesd_C_A,all,Anger_C_A_ayer,NaN,NaN


## 6) Comprobación rápida de rutas

Verificamos que los paths del manifest apuntan a archivos reales en disco.  
Esto ayuda a detectar rutas mal construidas o audios que faltan antes de entrenar.

In [6]:
from pathlib import Path
missing = manifest_es["path"].apply(lambda p: not Path(p).exists()).sum()
print("Missing paths:", missing)

print("Ejemplos:")
print(manifest_es.sample(min(10, len(manifest_es)), random_state=42)[["dataset","label","speaker","path"]])

Missing paths: 0
Ejemplos:
      dataset    label       speaker  \
365   mesd_es     fear      mesd_F_B   
393   mesd_es     fear      mesd_M_A   
1293   ses_sd  sadness  sessd_14_TAI   
1350   ses_sd     fear  sessd_16_TAI   
2102   ses_sd  sadness  sessd_39_TAI   
2269   ses_sd  sadness  sessd_44_IEO   
2346   ses_sd  neutral  sessd_46_TSI   
2215   ses_sd  sadness  sessd_42_IWW   
650   mesd_es  neutral      mesd_F_B   
298   mesd_es     fear      mesd_C_A   

                                                   path  
365   C:\Users\Rafa\Downloads\proyecto-ia-20260212T1...  
393   C:\Users\Rafa\Downloads\proyecto-ia-20260212T1...  
1293  C:\Users\Rafa\Downloads\proyecto-ia-20260212T1...  
1350  C:\Users\Rafa\Downloads\proyecto-ia-20260212T1...  
2102  C:\Users\Rafa\Downloads\proyecto-ia-20260212T1...  
2269  C:\Users\Rafa\Downloads\proyecto-ia-20260212T1...  
2346  C:\Users\Rafa\Downloads\proyecto-ia-20260212T1...  
2215  C:\Users\Rafa\Downloads\proyecto-ia-20260212T1...  
650   C:\U

## 7) Resumen del dataset generado

Mostramos un resumen del manifest final:
- número total de filas
- distribución por `dataset`
- distribución por `label`

Este paso sirve para comprobar balanceo y confirmar que se han incluido los datasets esperados.

In [7]:
print("Total filas:", len(manifest_es))
print("\nPor dataset:")
print(manifest_es["dataset"].value_counts())

print("\nPor label:")
print(manifest_es["label"].value_counts())

Total filas: 2738

Por dataset:
dataset
ses_sd         1588
mesd_es         862
mesd_es_emb     288
Name: count, dtype: int64

Por label:
label
disgust    457
fear       457
sadness    457
anger      456
joy        456
neutral    455
Name: count, dtype: int64
